# LecGap Phase 3 — Fine-tune prerequisite classifier (GPU)

This notebook fine-tunes a cross-encoder transformer over **LectureBank 1.0** prerequisite pairs and evaluates it with the same nested 5-fold CV used locally.

**Input (Kaggle dataset, name `lecturebank`):** place these two CSVs in `/kaggle/input/lecturebank/`
    - `prerequisite_annotation.csv` — `(Source_Topic_ID, Target_Topic_ID, If_prerequisite)`
    - `208topics.csv` — `(id, Topic, Topic_Link)`

**Output:** the fine-tuned model is written to `/kaggle/working/model/` — download it (as `model/*`) and load it on CPU for inference in the LecGap pipeline.

Set **Accelerator = GPU T4** and **Internet = On**.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
!pip install -q transformers sentence-transformers datasets scikit-learn

### Running the training + evaluation

This uses the same `scripts/kaggle_fine_tune.py` module from the repository. Paste it into the cell below, or upload it and run `!python /kaggle/working/kaggle_fine_tune.py ...`.

In [ ]:
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/AyushDevadiga1/lecture-comprehension-gap-detector/main"
files = {
    "kaggle_fine_tune.py": f"{BASE}/scripts/kaggle_fine_tune.py",
    "fine_tune.py": f"{BASE}/backend/pipeline/fine_tune.py",
}
for name, url in files.items():
    try:
        data = urllib.request.urlopen(url, timeout=30).read().decode()
        if name == "fine_tune.py":
            out = "/kaggle/working/backend/pipeline/fine_tune.py"
            os.makedirs(os.path.dirname(out), exist_ok=True)
        else:
            out = "/kaggle/working/" + name
        open(out, "w").write(data)
        print("fetched", name, "->", out)
    except Exception as e:
        print("Could not fetch", name, ":", e)
print("Done. If any fetch failed, paste the file contents manually and re-run.")

In [ ]:
!python /kaggle/working/kaggle_fine_tune.py \
    --input-dir /kaggle/input/lecturebank \
    --output-dir /kaggle/working/model \
    --epochs 3 --batch-size 32 --lr 2e-5 --max-neg-ratio 8

print('\nFine-tuned model is in /kaggle/working/model/ — download it for local CPU inference.')